A hyper-parameter is a parameter whose value is chosen by the egnineer (its dynamic)
Multi-Layer Perceptron 

In [60]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [61]:
# read in all the words
words = open("names.txt", "r").read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [62]:
len(words)

32033

In [63]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)} #creating Lookup table -string to integer mapping, +1 added so special character can have index position 0
stoi['.'] = 0 #special character
itos = {i:s for s,i in stoi.items()} #integer to string mapping - inverse of stoi 
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [64]:
# build the dataset

block_size = 3 #context length: how many characters do we take to predict the next one?
X,Y = [], []

for w in words[:5]:

    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '----->', itos[ix])
        context = context[1:] + [ix] #crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... -----> e
..e -----> m
.em -----> m
emm -----> a
mma -----> .
olivia
... -----> o
..o -----> l
.ol -----> i
oli -----> v
liv -----> i
ivi -----> a
via -----> .
ava
... -----> a
..a -----> v
.av -----> a
ava -----> .
isabella
... -----> i
..i -----> s
.is -----> a
isa -----> b
sab -----> e
abe -----> l
bel -----> l
ell -----> a
lla -----> .
sophia
... -----> s
..s -----> o
.so -----> p
sop -----> h
oph -----> i
phi -----> a
hia -----> .


In [65]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [66]:
C = torch.randn((27,2))

In [67]:
emb = C[X]   #embed all of the integers of X into C to create lookup table
emb.shape

torch.Size([32, 3, 2])

In [68]:
w1 = torch.randn((6,100))
b1 = torch.randn(100)

In [ ]:
h = torch.tanh(emb.view(-1,6) @ w1 + b1) 
# .view is to transform the matrix from a [32, 3, 2] to [32, 6] so we can perfrom multiplication
# tanh is to perfrom softmax and make the negative numbers positive


In [70]:
h.shape

torch.Size([32, 100])

In [71]:
w2 =torch.randn((100,27))
b2 =torch.randn(27)

In [72]:
logits = h @ w2 + b2

In [73]:
logits.shape

torch.Size([32, 27])

In [74]:
counts = logits.exp()

In [75]:
prob = counts / counts.sum(1, keepdims=True)

In [76]:
prob.shape

torch.Size([32, 27])

In [78]:
loss = -prob[torch.arange(32), Y].log().mean()    # negative log likelihood loss
loss

tensor(18.7612)

In [79]:
# ----------------now the whole code so far arranged ---------------------

In [80]:
X.shape, Y.shape

(torch.Size([32, 3]), torch.Size([32]))

In [85]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2), generator=g)
w1 = torch.randn((6,100), generator=g)
b1 = torch.randn(100, generator=g)
w2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, w1, b1, w2, b2]

In [86]:
sum(p.nelement() for p in parameters ) # number of parameters in total

3481

In [91]:
for p in parameters:
    p.requires_grad=True

counts  = logits.exp()
prob = counts / counts.sum(1, keepdims=True)
loss = -prob[torch.arange(32), Y].log().mean()
The above 3 lines is known as classifications and can be done directly by using the pytorch function  cross_entropy

In [ ]:
for _ in range(10):
    #Forward pass
    emb = C[X] #[32, 3, 2]
    h =  torch.tanh(emb.view(-1, 6) @ w1 + b1)  #(32, 100)
    logits = h @  w2 + b2 #(32, 27)
    loss  =  F.cross_entropy(logits, Y)

    # backward pass
    for p in parameters:
        p.grad = None   #same as setting to 0
    loss.backward()     #to populate the gradients

    # update
    for p in parameters:
        p.data += -0.1 * p.grad
        
print(loss.item())

13.656402587890625
11.298768997192383
9.4524564743042
7.984262943267822
6.891320705413818
6.100014686584473
5.452036380767822
4.898151874542236
4.414663791656494
3.985849380493164
